# Exploratory Data Analysis — Amazon Electronics Reviews

This notebook explores the Amazon Reviews 2023 (Electronics) dataset loaded into PostgreSQL.
It covers: data overview, rating distributions, temporal trends, NER extraction samples, and text characteristics.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

engine = create_engine(
    f"postgresql://{os.getenv('POSTGRES_USER', 'postgres')}:{os.getenv('POSTGRES_PASSWORD', 'postgres')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}"
    f"/{os.getenv('POSTGRES_DB', 'product_intelligence')}"
)

## 1. Data Overview

In [ ]:
review_count = pd.read_sql("SELECT COUNT(*) as count FROM reviews", engine).iloc[0, 0]
product_count = pd.read_sql("SELECT COUNT(*) as count FROM products", engine).iloc[0, 0]
print(f"Total reviews: {review_count:,}")
print(f"Total products: {product_count:,}")

## 2. Rating Distribution

In [ ]:
ratings = pd.read_sql("SELECT rating, COUNT(*) as count FROM reviews GROUP BY rating ORDER BY rating", engine)

fig, ax = plt.subplots()
ax.bar(ratings["rating"], ratings["count"], color="steelblue", edgecolor="white")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of Reviews")
ax.set_title("Rating Distribution")
ax.set_xticks([1, 2, 3, 4, 5])
for i, row in ratings.iterrows():
    ax.text(row["rating"], row["count"], f"{row['count']:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Reviews Over Time

In [ ]:
time_df = pd.read_sql(
    "SELECT DATE_TRUNC('month', timestamp) as month, COUNT(*) as count FROM reviews WHERE timestamp IS NOT NULL GROUP BY month ORDER BY month",
    engine,
)

fig, ax = plt.subplots()
ax.plot(time_df["month"], time_df["count"], color="steelblue", linewidth=1.5)
ax.set_xlabel("Month")
ax.set_ylabel("Number of Reviews")
ax.set_title("Reviews Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Top 20 Most Reviewed Products

In [ ]:
top_products = pd.read_sql(
    """
    SELECT r.asin, p.title, COUNT(*) as review_count, ROUND(AVG(r.rating)::numeric, 2) as avg_rating
    FROM reviews r
    LEFT JOIN products p ON r.parent_asin = p.parent_asin
    GROUP BY r.asin, p.title
    ORDER BY review_count DESC
    LIMIT 20
    """,
    engine,
)
top_products

## 5. Word Cloud of Negative Reviews (1-2 stars)

In [ ]:
from wordcloud import WordCloud

negative = pd.read_sql(
    "SELECT text FROM reviews WHERE rating <= 2 AND text IS NOT NULL LIMIT 10000",
    engine,
)

all_text = " ".join(negative["text"].tolist())
wc = WordCloud(width=1000, height=500, background_color="white", colormap="Reds", max_words=100).generate(all_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — Negative Reviews (1-2 Stars)", fontsize=16)
plt.tight_layout()
plt.show()

## 6. Sample NER Extractions

In [ ]:
from src.features.ner_extractor import load_nlp, extract_from_text

nlp = load_nlp()

sample_reviews = pd.read_sql(
    "SELECT text FROM reviews WHERE rating <= 2 AND text IS NOT NULL AND LENGTH(text) > 50 LIMIT 20",
    engine,
)

for _, row in sample_reviews.iterrows():
    entities = extract_from_text(nlp, row["text"])
    if entities["components"] or entities["issues"]:
        print(f"Review: {row['text'][:120]}...")
        print(f"  Components: {entities['components']}")
        print(f"  Issues: {entities['issues']}")
        print()

## 7. Review Length Distribution

In [ ]:
lengths = pd.read_sql(
    "SELECT LENGTH(text) as text_length FROM reviews WHERE text IS NOT NULL LIMIT 100000",
    engine,
)

fig, ax = plt.subplots()
ax.hist(lengths["text_length"].clip(upper=2000), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("Review Length (characters)")
ax.set_ylabel("Count")
ax.set_title("Review Length Distribution (capped at 2000 chars)")
plt.tight_layout()
plt.show()

print(f"Median length: {lengths['text_length'].median():.0f} chars")
print(f"Mean length: {lengths['text_length'].mean():.0f} chars")

## 8. Helpful Votes Distribution

In [ ]:
helpful = pd.read_sql(
    "SELECT helpful_votes FROM reviews WHERE helpful_votes IS NOT NULL AND helpful_votes > 0 LIMIT 100000",
    engine,
)

fig, ax = plt.subplots()
ax.hist(helpful["helpful_votes"].clip(upper=50), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("Helpful Votes")
ax.set_ylabel("Count")
ax.set_title("Helpful Votes Distribution (capped at 50)")
plt.tight_layout()
plt.show()

print(f"Reviews with >0 helpful votes: {len(helpful):,}")
print(f"Median helpful votes (when >0): {helpful['helpful_votes'].median():.0f}")